In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())


ROOT: /content/drive/MyDrive/MaintainAI/code
Exists: True
src exists: True


# 02 — FD001 dataset analysis
Calls `src/data_cmapss.py`. Findings feed `docs/DATASET.md`.

In [5]:
from src.data_cmapss import load_fd001, add_rul, SENSOR_COLS
tr, te, rul = load_fd001('CMAPSSData')
print('train', tr.shape, '| test', te.shape, '| rul', rul.shape)
print('train engines', tr['unit'].nunique(), '| test engines', te['unit'].nunique())
print(tr.groupby('unit')['cycle'].max().describe().round(1))
print('test RUL:', rul.describe().round(1).to_dict())
print('sensor std (constant ~0):')
print(tr[SENSOR_COLS].std().round(4).to_string())

train (20631, 26) | test (13096, 26) | rul (100,)
train engines 100 | test engines 100
count    100.0
mean     206.3
std       46.3
min      128.0
25%      177.0
50%      199.0
75%      229.2
max      362.0
Name: cycle, dtype: float64
test RUL: {'count': 100.0, 'mean': 75.5, 'std': 41.8, 'min': 7.0, '25%': 32.8, '50%': 86.0, '75%': 112.2, 'max': 145.0}
sensor std (constant ~0):
s1      0.0000
s2      0.5001
s3      6.1311
s4      9.0006
s5      0.0000
s6      0.0014
s7      0.8851
s8      0.0710
s9     22.0829
s10     0.0000
s11     0.2671
s12     0.7376
s13     0.0719
s14    19.0762
s15     0.0375
s16     0.0000
s17     1.5488
s18     0.0000
s19     0.0000
s20     0.1807
s21     0.1083


In [6]:
# Deeper analysis: RUL distribution on train (capped at 125)
from src.data_cmapss import add_capped_rul
tr = add_capped_rul(add_rul(tr), cap=125)
print('Train RUL (capped 125):', tr['rul_capped'].describe().round(1).to_dict())
print('Train RUL (uncapped):', tr['rul'].describe().round(1).to_dict())
print('Max cycle per engine:', tr.groupby('unit')['cycle'].max().describe().round(1))

Train RUL (capped 125): {'count': 20631.0, 'mean': 86.8, 'std': 41.7, 'min': 0.0, '25%': 51.0, '50%': 103.0, '75%': 125.0, 'max': 125.0}
Train RUL (uncapped): {'count': 20631.0, 'mean': 107.8, 'std': 68.9, 'min': 0.0, '25%': 51.0, '50%': 103.0, '75%': 155.0, 'max': 361.0}
Max cycle per engine: count    100.0
mean     206.3
std       46.3
min      128.0
25%      177.0
50%      199.0
75%      229.2
max      362.0
Name: cycle, dtype: float64


In [7]:
# Check constant sensors (should have near-zero std)
sensor_std = tr[SENSOR_COLS].std().round(6)
constant = sensor_std[sensor_std < 0.01].index.tolist()
print('Near-constant sensors:', constant)
print('Informative sensors:', [s for s in SENSOR_COLS if s not in constant])

Near-constant sensors: ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
Informative sensors: ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


In [8]:
# Engine-level split verification
from src.data_cmapss import engine_split, assert_disjoint
units = sorted(int(u) for u in tr['unit'].unique())
tr_units, va_units = engine_split(units, 0.2, 42)
print(f'Train engines: {len(tr_units)}, Val engines: {len(va_units)}')
print('Train units:', tr_units[:10], '...')
print('Val units:', va_units)
assert_disjoint(tr_units, va_units)
print('✓ Train/Val engine split is disjoint')

Train engines: 80, Val engines: 20
Train units: [31, 22, 33, 97, 81, 50, 84, 27, 88, 34] ...
Val units: [43, 42, 92, 10, 66, 51, 2, 71, 16, 79, 74, 11, 56, 57, 73, 46, 49, 93, 77, 38]
✓ Train/Val engine split is disjoint


In [9]:
# Test set: verify unit IDs are 1-100 but distinct fleet from train
test_units = sorted(int(u) for u in te['unit'].unique())
print('Test units:', test_units[:10], '...')
print('Test engines:', len(test_units))
print('RUL file length:', len(rul))
print('RUL matches test engines:', len(rul) == len(test_units))

Test units: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] ...
Test engines: 100
RUL file length: 100
RUL matches test engines: True


## Summary for `docs/DATASET.md`
- **Source**: NASA PCoE CMAPSS Jet Engine Simulated Data
- **FD001 scope**: 100 train + 100 test engines; single operating condition; single fault mode (HPC degradation)
- **Train**: 20,631 rows, max cycle 362, mean RUL 107.8 (capped-125 mean 86.8)
- **Test**: 13,096 rows, 100 engines, true RUL mean 75.5 (min 7, max 145)
- **Constant sensors** (std≈0): s1, s5, s6, s10, s16, s18, s19 → exclude from features
- **Leakage policy**: Split by ENGINE (unit), never by row; test fleet IDs overlap train but are distinct
- **Target**: Remaining Useful Life (RUL) in cycles; piecewise-linear capped RUL (cap=125) for training